In [ ]:
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from pyvis.network import Network
from langchain.chat_models import init_chat_model
import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from typing import Tuple, Optional
import re
import numpy as np
from sentence_transformers import SentenceTransformer, util

from dotenv import load_dotenv
import os
import asyncio

In [ ]:
load_dotenv()
os.environ["DEEPSEEK_API_KEY"] = os.getenv("DEEPSEEK_API_KEY")
os.environ["QWEN_API_KEY"] = os.getenv("QWEN_API_KEY")

In [61]:
llm_ds=init_chat_model(
    model='deepseek:deepseek-chat',
    temperature=1.3)

In [9]:
llm_qwen=ChatOpenAI(
    model="qwen-max",
    openai_api_key=os.getenv("QWEN_API_KEY"),
    openai_api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=1.3
)

# 1 相关性

In [3]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

In [4]:
def build_full_question(row):
    """
    根据字段拼合完整试题（题干 + 正确答案文本）
    输入 row: dict，包含字段：
        '题干', '选项', '正确答案'
    输出: 完整试题字符串
    """
    question = row['题干'].strip()
    answer_raw = row['正确答案'].strip()
    options_text = row['选项'].strip()

    # 1. 尝试从选项字段中提取正确答案对应的文本（选择题）
    answer_text = answer_raw
    if re.match(r'^[A-D]$', answer_raw) and options_text:
        # 按 "A." "B." 等分割，简单提取
        parts = re.split(r'[A-D]\.', options_text)
        # parts[0] 是空，parts[1] 对应 A 选项，parts[2] 对应 B，依次类推
        opt_dict = {}
        opt_letters = re.findall(r'([A-D])\.', options_text)
        opt_contents = re.split(r'[A-D]\.', options_text)[1:]  # 去掉首空
        for letter, content in zip(opt_letters, opt_contents):
            opt_dict[letter] = content.strip()
        if answer_raw in opt_dict:
            answer_text = opt_dict[answer_raw]

    # 2. 处理括号占位符（如“（）”）
    if '（）' in question or '（ ）' in question:
        full = question.replace('（）', answer_text).replace('（ ）', answer_text)
    else:
        full = f"{question} 正确答案：{answer_text}"
    return full

In [5]:
def compute_knowledge_relevance(full_question, knowledge_context):
    """
    计算完整试题与知识上下文的语义相关性得分
    策略：将知识上下文按换行拆分为句子列表，计算每句与完整试题的余弦相似度，
          取相似度最高的 top-3 句的平均值（若少于3句则全取平均）。
    返回: 浮点数相关性得分
    """
    if not knowledge_context or pd.isna(knowledge_context):
        return 0.0
    # 按换行分割，过滤空行
    sentences = [s.strip() for s in knowledge_context.split('。') if s.strip()]
    if not sentences:
        return 0.0

    # 编码
    q_emb = embedder.encode(full_question, convert_to_tensor=True)
    s_embs = embedder.encode(sentences, convert_to_tensor=True)

    # 计算余弦相似度
    cos_scores = util.cos_sim(q_emb, s_embs)[0].cpu().numpy()

    # 取 top-3 平均值（或全部平均如果少于3）
    k = min(3, len(sentences))
    top_k_idx = np.argsort(cos_scores)[-k:]
    relevance = cos_scores[top_k_idx].mean()
    return float(relevance)

In [6]:
def evaluate_relevance_from_txt(input_txt_path, output_txt_path):
    """
    主函数：读取 .txt 文件，每行格式为分号分隔的字段：
        知识点|难度|题型|是否涉及运算|认知层级|题干|选项|正确答案|知识上下文
    追加一列“知识点相关性”并输出新 .txt
    """
    import pandas as pd

    # 读取所有行
    with open(input_txt_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    header = lines[0].split(';')
    # 确保字段顺序正确，这里我们按顺序硬编码字段名
    field_names = ['知识点', '难度', '题型', '是否运算', '认知层级',
                   '题干', '选项', '正确答案', '知识上下文']

    results = []
    relevance_scores = []

    for i, line in enumerate(lines):
        parts = line.split(';')
        if len(parts) < 9:
            print(f"第{i+1}行字段不足，跳过")
            continue
        # 构造行字典
        row = {name: parts[idx] for idx, name in enumerate(field_names)}

        # 1. 拼合完整试题
        full_q = build_full_question(row)

        # 2. 计算相关性
        score = compute_knowledge_relevance(full_q, row['知识上下文'])
        relevance_scores.append(score)

        # 3. 构造新行：原字段 + 新字段
        new_line = line + f";{score:.4f}"
        results.append(new_line)

    # 打印得分列表
    print("知识点相关性得分列表：")
    print([round(s, 4) for s in relevance_scores])

    # 写入新文件
    with open(output_txt_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(results))

    print(f"处理完成，结果已保存至：{output_txt_path}")

In [65]:
# ========== 使用示例 ==========
filename="./output/comparison/raw/qwen-210-noncontext.txt"
evaluate_relevance_from_txt(filename, filename[:-4]+'_rel'+'.txt')

知识点相关性得分列表：
[0.5821, 0.6117, 0.6655, 0.5516, 0.661, 0.6416, 0.5986, 0.7109, 0.7039, 0.6852, 0.5256, 0.7409, 0.6778, 0.6722, 0.6301, 0.6884, 0.6379, 0.6035, 0.7298, 0.6157, 0.7306, 0.6014, 0.6222, 0.5895, 0.6065, 0.6418, 0.686, 0.6433, 0.6748, 0.6422, 0.7331, 0.721, 0.6977, 0.6731, 0.6308, 0.6676, 0.6322, 0.6888, 0.7139, 0.663, 0.7109, 0.7226, 0.701, 0.6967, 0.6616, 0.684, 0.6757, 0.6926, 0.6861, 0.6818, 0.6389, 0.6584, 0.6254, 0.6082, 0.708, 0.6692, 0.7128, 0.5935, 0.684, 0.7168, 0.6642, 0.6467, 0.6263, 0.6589, 0.714, 0.5828, 0.7064, 0.6643, 0.6856, 0.6828, 0.6964, 0.7423, 0.5722, 0.6424, 0.746, 0.7179, 0.6666, 0.6254, 0.7108, 0.7294, 0.651, 0.6245, 0.6764, 0.7116, 0.6959, 0.6521, 0.6507, 0.6524, 0.6957, 0.6031, 0.682, 0.5469, 0.6801, 0.7061, 0.7219, 0.6859, 0.6544, 0.6752, 0.6024, 0.6988]
处理完成，结果已保存至：./output/comparison/raw/qwen-210-noncontext_rel.txt


# 2 可解释性

In [11]:
def eval_answer(llm,input_txt_path: str, output_txt_path: str, n: int = 5):
    """
    对 .txt 文件中的每道试题，调用 LLM 求解 n 次，记录每次的答案（选项字母），
    追加在原行末尾，以分号分隔，输出新文件。

    输入文件格式（无表头，分号分隔）：
        字段1:知识点, 字段2:难度, 字段3:题型, 字段4:是否运算, 字段5:认知层级,
        字段6:题干, 字段7:选项, 字段8:正确答案, 字段9:知识上下文, 字段10...:其他已有标签
    输出文件格式：原行 + ;答案1;答案2;...;答案n
    """
    with open(input_txt_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    output_lines = []

    for line in lines:
        parts = line.split(';')
        # 提取题干和选项（根据示例：第6字段题干，第7字段选项）
        question_text = parts[5] if len(parts) > 5 else ""
        options_text = parts[6] if len(parts) > 6 else ""

        # 构建完整的选择题文本
        full_question = f"{question_text}\n{options_text}"

        # 为 LLM 构建系统提示和用户提示
        system_prompt = '''
        你是一名资深地理教师，请回答以下选择题。
        只输出选项字母（A、B、C、D）中的一个，不要输出任何其他内容。
        '''
        user_prompt = f"题目：{full_question}\n你的答案："

        
        print(f"\n第{len(output_lines)+1}题",end=' ')
        answers = []
        for i in range(n):
            print(f"第{i+1}次",end=' | ')
            try:
                # 调用已定义好的 llm 变量
                response = llm.invoke([
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ])
                # 获取模型输出文本
                if hasattr(response, 'content'):
                    content = response.content.strip()
                else:
                    content = str(response).strip()
                # 提取第一个 A-D 字母
                match = re.search(r'[A-D]', content)
                answer = match.group(0) if match else 'ERROR'
            except Exception as e:
                print(f"第 {len(output_lines)+1} 行第 {i+1} 次调用出错: {e}")
                answer = 'ERROR'
            
            answers.append(answer)

        # 原行 + 追加的 n 个答案字段
        new_line = line + ';' + ';'.join(answers)
        output_lines.append(new_line)

    # 写入输出文件
    with open(output_txt_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(output_lines))

    print(f"处理完成，共 {len(output_lines)} 道题，每道题记录 {n} 次答案。")
    print(f"输出文件：{output_txt_path}")

In [63]:
filename2="./output/comparison/raw/questions_220_qwen_rel.txt"
eval_answer(llm_ds,filename2, filename2[:-4]+'_ans'+'.txt', n=5)


第1题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第2题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第3题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第4题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第5题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第6题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第7题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第8题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第9题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第10题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第11题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第12题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第13题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第14题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第15题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第16题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第17题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第18题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第19题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第20题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第21题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第22题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第23题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第24题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第25题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第26题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第27题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第28题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 


In [64]:
filename2="./output/comparison/raw/questions_230_qwen_rel.txt"
eval_answer(llm_ds,filename2, filename2[:-4]+'_ans'+'.txt', n=5)


第1题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第2题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第3题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第4题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第5题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第6题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第7题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第8题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第9题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第10题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第11题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第12题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第13题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第14题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第15题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第16题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第17题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第18题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第19题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第20题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第21题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第22题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第23题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第24题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第25题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第26题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第27题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第28题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 


In [68]:
# ========== 使用示例 ==========
filename="./output/comparison/raw/qwen-210-noncontext.txt"
evaluate_relevance_from_txt(filename, filename[:-4]+'_rel'+'.txt')
filename2="./output/comparison/raw/qwen-210-noncontext_rel.txt"
eval_answer(llm_ds,filename2, filename2[:-4]+'_ans'+'.txt', n=5)

知识点相关性得分列表：
[0.5821, 0.6117, 0.6655, 0.5516, 0.661, 0.6416, 0.5986, 0.7109, 0.7039, 0.6852, 0.5256, 0.7409, 0.6778, 0.6722, 0.6301, 0.6884, 0.6379, 0.6035, 0.7298, 0.6157, 0.7306, 0.6014, 0.6222, 0.5895, 0.6065, 0.6418, 0.686, 0.6433, 0.6748, 0.6422, 0.7331, 0.721, 0.6977, 0.6731, 0.6308, 0.6676, 0.6322, 0.6888, 0.7139, 0.663, 0.7109, 0.7226, 0.701, 0.6967, 0.6616, 0.684, 0.6757, 0.6926, 0.6861, 0.6818, 0.6389, 0.6584, 0.6254, 0.6082, 0.708, 0.6692, 0.7128, 0.5935, 0.684, 0.7168, 0.6642, 0.6467, 0.6263, 0.6589, 0.714, 0.5828, 0.7064, 0.6643, 0.6856, 0.6828, 0.6964, 0.7423, 0.5722, 0.6424, 0.746, 0.7179, 0.6666, 0.6254, 0.7108, 0.7294, 0.651, 0.6245, 0.6764, 0.7116, 0.6959, 0.6521, 0.6507, 0.6524, 0.6957, 0.6031, 0.682, 0.5469, 0.6801, 0.7061, 0.7219, 0.6859, 0.6544, 0.6752, 0.6024, 0.6988]
处理完成，结果已保存至：./output/comparison/raw/qwen-210-noncontext_rel.txt

第1题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第2题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第3题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第4题 第1次 | 第2次 | 第3次 | 第4次 | 第

In [71]:
# ========== 使用示例 ==========
filename="./output/comparison/raw/qwen-220-noncontext.txt"
evaluate_relevance_from_txt(filename, filename[:-4]+'_rel'+'.txt')
filename2="./output/comparison/raw/qwen-220-noncontext_rel.txt"
eval_answer(llm_ds,filename2, filename2[:-4]+'_ans'+'.txt', n=5)

知识点相关性得分列表：
[0.6058, 0.6923, 0.5339, 0.7274, 0.7143, 0.6648, 0.612, 0.5858, 0.618, 0.6174, 0.5494, 0.6145, 0.6128, 0.6809, 0.6817, 0.5971, 0.6802, 0.7108, 0.6389, 0.607, 0.6991, 0.7084, 0.6918, 0.6682, 0.6929, 0.709, 0.6074, 0.6631, 0.6116, 0.6315, 0.7162, 0.703, 0.6859, 0.6258, 0.7244, 0.6993, 0.6976, 0.6937, 0.617, 0.6875, 0.7039, 0.6128, 0.6697, 0.6599, 0.5528, 0.5891, 0.5821, 0.7005, 0.7071, 0.6334, 0.7054, 0.6448, 0.6303, 0.7297, 0.6703, 0.6692, 0.6374, 0.6997, 0.7108, 0.6961, 0.6381, 0.7208, 0.5763, 0.6639, 0.6841, 0.6096, 0.6666, 0.6829, 0.6333, 0.725, 0.6578, 0.6791, 0.6603, 0.6649, 0.6409, 0.6329, 0.7344, 0.6607, 0.5972, 0.7327, 0.694, 0.5974, 0.5536, 0.6256, 0.6772, 0.5881, 0.703, 0.6927, 0.7382, 0.706, 0.7297, 0.7014, 0.668, 0.7113, 0.7035, 0.7096, 0.7189, 0.6986, 0.6199, 0.6347]
处理完成，结果已保存至：./output/comparison/raw/qwen-220-noncontext_rel.txt

第1题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第2题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第3题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第4题 第1次 | 第2次 | 第3次 | 第4次 | 

In [72]:
# ========== 使用示例 ==========
filename="./output/comparison/raw/qwen-230-noncontext.txt"
evaluate_relevance_from_txt(filename, filename[:-4]+'_rel'+'.txt')
filename2="./output/comparison/raw/qwen-230-noncontext_rel.txt"
eval_answer(llm_ds,filename2, filename2[:-4]+'_ans'+'.txt', n=5)

知识点相关性得分列表：
[0.7471, 0.7086, 0.6863, 0.6961, 0.6903, 0.6565, 0.6728, 0.6529, 0.705, 0.6359, 0.6709, 0.629, 0.7094, 0.6427, 0.6126, 0.6905, 0.6825, 0.6501, 0.7177, 0.6565, 0.6789, 0.6281, 0.6257, 0.6308, 0.6802, 0.6624, 0.6869, 0.6106, 0.6458, 0.7103, 0.6742, 0.6661, 0.6172, 0.6498, 0.6743, 0.7101, 0.6646, 0.6744, 0.7082, 0.6704, 0.6884, 0.6527, 0.7097, 0.6676, 0.6707, 0.7049, 0.6738, 0.664, 0.6884, 0.6587, 0.7036, 0.6288, 0.6932, 0.6422, 0.6776, 0.721, 0.6753, 0.5096, 0.7057, 0.711, 0.6422, 0.7131, 0.633, 0.699, 0.6869, 0.7093, 0.6999, 0.6395, 0.6676, 0.6585, 0.6824, 0.6767, 0.6632, 0.7294, 0.6274, 0.678, 0.6881, 0.6736, 0.6748, 0.7404, 0.718, 0.7074, 0.687, 0.6831, 0.5678, 0.6777, 0.6624, 0.645, 0.725, 0.6958, 0.5816, 0.7412, 0.6637, 0.7368, 0.7667, 0.7139, 0.6703, 0.6452, 0.6969, 0.7616]
处理完成，结果已保存至：./output/comparison/raw/qwen-230-noncontext_rel.txt

第1题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第2题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第3题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第4题 第1次 | 第2次 | 第3次 | 第4次 | 第

In [73]:

# ========== 使用示例 ==========
filename="./output/comparison/raw/qwen-210-text-rag.txt"
evaluate_relevance_from_txt(filename, filename[:-4]+'_rel'+'.txt')
filename2="./output/comparison/raw/qwen-210-text-rag_rel.txt"
eval_answer(llm_ds,filename2, filename2[:-4]+'_ans'+'.txt', n=5)

知识点相关性得分列表：
[0.6367, 0.6594, 0.6565, 0.6828, 0.7835, 0.7199, 0.6016, 0.8383, 0.7373, 0.6319, 0.6702, 0.8091, 0.6029, 0.6588, 0.6151, 0.6959, 0.6481, 0.6692, 0.6579, 0.6593, 0.6916, 0.656, 0.7289, 0.6054, 0.7502, 0.7476, 0.7374, 0.6711, 0.5754, 0.6937, 0.7051, 0.7352, 0.6683, 0.6462, 0.6961, 0.6065, 0.6631, 0.6176, 0.7254, 0.6306, 0.7177, 0.7094, 0.6849, 0.5499, 0.6752, 0.7229, 0.7612, 0.6581, 0.8311, 0.6026, 0.7058, 0.7099, 0.7023, 0.7413, 0.8595, 0.7302, 0.8696, 0.6336, 0.558, 0.7789, 0.6962, 0.7026, 0.6506, 0.7586, 0.6607, 0.709, 0.8437, 0.6464, 0.7818, 0.6757, 0.6748, 0.597, 0.7557, 0.5756, 0.73, 0.7591, 0.735, 0.6982, 0.8126, 0.8044, 0.6405, 0.6765, 0.6916, 0.8496, 0.589, 0.8533, 0.8361, 0.6631, 0.676, 0.8493, 0.7238, 0.6137, 0.7469, 0.6475, 0.6972, 0.6889, 0.7118, 0.7449, 0.7083, 0.7438]
处理完成，结果已保存至：./output/comparison/raw/qwen-210-text-rag_rel.txt

第1题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第2题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第3题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第4题 第1次 | 第2次 | 第3次 | 第4次 | 

In [74]:

# ========== 使用示例 ==========
filename="./output/comparison/raw/qwen-220-text-rag.txt"
evaluate_relevance_from_txt(filename, filename[:-4]+'_rel'+'.txt')
filename2="./output/comparison/raw/qwen-220-text-rag_rel.txt"
eval_answer(llm_ds,filename2, filename2[:-4]+'_ans'+'.txt', n=5)

知识点相关性得分列表：
[0.7499, 0.7116, 0.5902, 0.7001, 0.6441, 0.6404, 0.6973, 0.7993, 0.7513, 0.7942, 0.7818, 0.7553, 0.7105, 0.7589, 0.6868, 0.7794, 0.6706, 0.6652, 0.7027, 0.713, 0.5192, 0.6915, 0.7478, 0.7443, 0.6387, 0.5865, 0.6094, 0.6268, 0.5463, 0.7042, 0.7157, 0.6674, 0.6671, 0.8106, 0.7685, 0.6713, 0.5376, 0.6587, 0.7585, 0.7401, 0.6408, 0.6455, 0.6392, 0.766, 0.6532, 0.69, 0.6829, 0.7956, 0.72, 0.6344, 0.6757, 0.6595, 0.6776, 0.6885, 0.7401, 0.6029, 0.7728, 0.669, 0.6614, 0.6404, 0.8266, 0.8189, 0.6621, 0.6757, 0.716, 0.7121, 0.7944, 0.7217, 0.6164, 0.7469, 0.7867, 0.7755, 0.6359, 0.7888, 0.6304, 0.7946, 0.5986, 0.6549, 0.7703, 0.8019, 0.7506, 0.6126, 0.5894, 0.7647, 0.5291, 0.7447, 0.7145, 0.6468, 0.7586, 0.6531, 0.664, 0.6686, 0.7693, 0.7484, 0.8085, 0.7354, 0.6575, 0.7616, 0.6557, 0.6366]
处理完成，结果已保存至：./output/comparison/raw/qwen-220-text-rag_rel.txt

第1题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第2题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第3题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第4题 第1次 | 第2次 | 第3次 | 第4次 | 

In [75]:

# ========== 使用示例 ==========
filename="./output/comparison/raw/qwen-230-text-rag.txt"
evaluate_relevance_from_txt(filename, filename[:-4]+'_rel'+'.txt')
filename2="./output/comparison/raw/qwen-230-text-rag_rel.txt"
eval_answer(llm_ds,filename2, filename2[:-4]+'_ans'+'.txt', n=5)

知识点相关性得分列表：
[0.6578, 0.6313, 0.8515, 0.7214, 0.8646, 0.5865, 0.6059, 0.6392, 0.6152, 0.7061, 0.642, 0.8372, 0.6716, 0.6956, 0.854, 0.8636, 0.8472, 0.7329, 0.6792, 0.7306, 0.6389, 0.5216, 0.7972, 0.7948, 0.8198, 0.5398, 0.6828, 0.727, 0.8517, 0.6582, 0.4985, 0.6193, 0.6708, 0.5187, 0.6544, 0.541, 0.6704, 0.6269, 0.7182, 0.4472, 0.7824, 0.8188, 0.6857, 0.7417, 0.6551, 0.87, 0.6893, 0.6667, 0.8679, 0.8314, 0.5396, 0.5999, 0.8256, 0.8569, 0.6962, 0.7713, 0.6825, 0.7427, 0.8367, 0.5825, 0.742, 0.5587, 0.5946, 0.6595, 0.617, 0.7156, 0.8045, 0.7179, 0.6663, 0.7167, 0.8874, 0.7614, 0.7287, 0.7464, 0.6697, 0.8012, 0.6726, 0.4906, 0.7389, 0.6808, 0.8301, 0.8727, 0.6132, 0.6739, 0.843, 0.6524, 0.6163, 0.8259, 0.7518, 0.7458, 0.8313, 0.6541, 0.6635, 0.5923, 0.8377, 0.7132, 0.7754, 0.8303, 0.6479, 0.6961]
处理完成，结果已保存至：./output/comparison/raw/qwen-230-text-rag_rel.txt

第1题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第2题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第3题 第1次 | 第2次 | 第3次 | 第4次 | 第5次 | 
第4题 第1次 | 第2次 | 第3次 | 第4次 | 

# 3 难度准确性

In [14]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

In [15]:
def eval_difficulty_accuracy(generated_txt, bank_xlsx, section, output_txt=None):
    """
    评估生成试题的难度准确性。
    参数:
        generated_txt : str  - 生成试题的 .txt 文件路径（分号分隔，无表头）
        bank_xlsx    : str  - 试题库 .xlsx 文件路径（包含表头，列顺序见说明）
        section      : str  - 生成试题所属的节（如“流水地貌”），用于筛选题库
        output_txt   : str  - 输出文件路径，默认为原文件名加“_dif”
    输出:
        新 .txt 文件，每行末尾追加一列难度准确性得分（相似度），保留4位小数。
    """
    if output_txt is None:
        output_txt = generated_txt.replace('.txt', '_dif.txt')

    # 加载语义相似度模型（CPU 可用）
    

    # ========== 读取题库 ==========
    df = pd.read_excel(bank_xlsx, dtype=str)  # 全部按字符串读取
    bank_items = []  # 每个元素为 (节, 难度, 向量)

    for idx, row in df.iterrows():
        # 按列索引取字段（根据图片列顺序）
        sec = str(row.iloc[1]) if pd.notna(row.iloc[1]) else ''      # 第2列：节
        diff = str(row.iloc[9]) if pd.notna(row.iloc[9]) else ''     # 第10列：难度
        # 组合完整试题文本：材料 + 提问 + 选项1~4
        parts = [
            str(row.iloc[2]) if pd.notna(row.iloc[2]) else '',  # 材料
            str(row.iloc[3]) if pd.notna(row.iloc[3]) else '',  # 提问
            str(row.iloc[4]) if pd.notna(row.iloc[4]) else '',  # 选项1
            str(row.iloc[5]) if pd.notna(row.iloc[5]) else '',  # 选项2
            str(row.iloc[6]) if pd.notna(row.iloc[6]) else '',  # 选项3
            str(row.iloc[7]) if pd.notna(row.iloc[7]) else '',  # 选项4
        ]
        full_text = ' '.join([p for p in parts if p.strip()])
        if not full_text.strip():
            continue  # 跳过空题
        # 计算归一化向量（点积即余弦相似度）
        emb = model.encode(full_text, normalize_embeddings=True)
        bank_items.append((sec, diff, emb))

    # ========== 处理生成试题 ==========
    with open(generated_txt, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    out_lines = []
    for line in lines:
        parts = line.split(';')
        if len(parts) < 7:
            print(f"警告：行字段不足，跳过 - {line[:50]}")
            out_lines.append(line + ';')  # 补一个空分号
            continue

        diff_gen = parts[1]                     # 生成难度（第2字段）
        q_text = parts[5]                       # 题干（第6字段）
        opts = parts[6]                          # 选项（第7字段）
        full_q = q_text + ' ' + opts

        q_emb = model.encode(full_q, normalize_embeddings=True)

        # 筛选题库中相同节且相同难度的题目
        candidates = [item for item in bank_items if item[0] == section and item[1] == diff_gen]
        if not candidates:
            score = 0.0
        else:
            # 计算与所有候选向量的余弦相似度，取最大值
            bank_embs = np.array([item[2] for item in candidates])
            sims = np.dot(bank_embs, q_emb)      # 所有点积（已归一化）
            score = float(np.max(sims))

        new_line = line + f';{score:.4f}'
        out_lines.append(new_line)

    # ========== 写入输出文件 ==========
    with open(output_txt, 'w', encoding='utf-8') as f:
        f.write('\n'.join(out_lines))

    print(f"难度准确性评估完成，结果已保存至：{output_txt}")

In [86]:
filename3="./output/comparison/raw/qwen-230-noncontext_rel_ans.txt"

eval_difficulty_accuracy(
    generated_txt=filename3,
    bank_xlsx='./data/题库 _已标注.xlsx',
    section='喀斯特、海岸和冰川地貌',      
    output_txt=filename3[:-4]+'_dif'+'.txt'   
    )

难度准确性评估完成，结果已保存至：./output/comparison/raw/qwen-230-noncontext_rel_ans_dif.txt


In [ ]:
block2id={'流水地貌':0,'风成地貌':1,'喀斯特、海岸和冰川地貌':2}

# 4 指标分析

In [ ]:
# 计算可解释性得分（答案一致率）
def compute_explainability(txt_path):
    """
    读取评估结果文件，计算每道题的可解释性得分（答案一致率）
    输入: txt文件路径
    输出: 每道题的可解释性得分列表
    """
    with open(txt_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    explainability_scores = []

    for line in lines:
        parts = line.split(';')
        # 字段16个，5次作答在索引10-14
        if len(parts) < 15:
            explainability_scores.append(0.0)
            continue

        answers = parts[10:15]
        valid_answers = [a for a in answers if a and a in 'ABCD']
        if not valid_answers:
            explainability_scores.append(0.0)
            continue

        from collections import Counter
        counter = Counter(valid_answers)
        most_common_count = counter.most_common(1)[0][1]
        score = most_common_count / len(valid_answers)
        explainability_scores.append(score)

    return explainability_scores

# 测试
test_file = './output/comparison/raw/ds-210-noncontext_rel_ans_dif.txt'
scores = compute_explainability(test_file)
print(f"可解释性得分列表（前10题）: {[round(s, 2) for s in scores[:10]]}")
print(f"平均可解释性得分: {sum(scores)/len(scores):.4f}" if scores else "无数据")

In [58]:
# 单个文件指标分析
import numpy as np
from collections import defaultdict

def analyze_single_file(txt_path):
    """
    分析单个评估结果文件的3个指标表现
    输入: txt文件路径
    输出: 打印汇总报告
    """
    with open(txt_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    # 调试：查看第一行字段数量
    if lines:
        parts = lines[0].split(';')
        print(f"字段数量={len(parts)}")

    # 按组别存储数据
    groups_data = defaultdict(lambda: {
        'relevance': [],
        'explainability': [],
        'accuracy': []
    })

    for line in lines:
        parts = line.split(';')
        # 字段16个：索引0-15，相关性在索引9，作答在10-14，难度准确性在15
        if len(parts) < 16:
            continue

        group = parts[0] if parts[0] else "未知"  # 组别
        relevance = float(parts[9]) if parts[9] else 0.0  # 相关性在索引9

        answers = parts[10:15]  # 5次作答在索引10-14
        valid_answers = [a for a in answers if a and a in 'ABCD']
        if valid_answers:
            from collections import Counter
            counter = Counter(valid_answers)
            most_common_count = counter.most_common(1)[0][1]
            exp_score = most_common_count / len(valid_answers)
        else:
            exp_score = 0.0

        accuracy = float(parts[15]) if parts[15] else 0.0  # 难度准确性在索引15

        groups_data[group]['relevance'].append(relevance)
        groups_data[group]['explainability'].append(exp_score)
        groups_data[group]['accuracy'].append(accuracy)

    # 输出报告
    print("="*60)
    print(f"文件: {txt_path}")
    print("="*60)

    for group, data in groups_data.items():
        rel = np.array(data['relevance'])
        exp = np.array(data['explainability'])
        acc = np.array(data['accuracy'])

        print(f"\n【{group}】 (共{len(rel)}题)")
        print(f"  相关性:     均值={rel.mean():.4f}, 标准差={rel.std():.4f}")
        print(f"  可解释性:   均值={exp.mean():.4f}, 标准差={exp.std():.4f}")
        print(f"  难度准确性: 均值={acc.mean():.4f}, 标准差={acc.std():.4f}")

    # 整体统计
    all_rel = []
    all_exp = []
    all_acc = []
    for data in groups_data.values():
        all_rel.extend(data['relevance'])
        all_exp.extend(data['explainability'])
        all_acc.extend(data['accuracy'])

    print(f"\n【整体】 (共{len(all_rel)}题)")
    print(f"  相关性:     均值={np.mean(all_rel):.4f}, 标准差={np.std(all_rel):.4f}")
    print(f"  可解释性:   均值={np.mean(all_exp):.4f}, 标准差={np.std(all_exp):.4f}")
    print(f"  难度准确性: 均值={np.mean(all_acc):.4f}, 标准差={np.std(all_acc):.4f}")

# 测试
test_file='./output/comparison/raw/ds-230-text-rag_rel_ans_dif.txt'
analyze_single_file(test_file)

字段数量=16
文件: ./output/comparison/raw/ds-230-text-rag_rel_ans_dif.txt

【岬角】 (共6题)
  相关性:     均值=0.6090, 标准差=0.0725
  可解释性:   均值=1.0000, 标准差=0.0000
  难度准确性: 均值=0.6584, 标准差=0.1166

【海滩】 (共6题)
  相关性:     均值=0.7979, 标准差=0.0499
  可解释性:   均值=1.0000, 标准差=0.0000
  难度准确性: 均值=0.6059, 标准差=0.1112

【喀斯特地貌】 (共2题)
  相关性:     均值=0.7577, 标准差=0.0012
  可解释性:   均值=1.0000, 标准差=0.0000
  难度准确性: 均值=0.7868, 标准差=0.0237

【湖南张家界黄龙洞】 (共4题)
  相关性:     均值=0.6165, 标准差=0.0350
  可解释性:   均值=1.0000, 标准差=0.0000
  难度准确性: 均值=0.5024, 标准差=0.1207

【角峰】 (共5题)
  相关性:     均值=0.6202, 标准差=0.0684
  可解释性:   均值=1.0000, 标准差=0.0000
  难度准确性: 均值=0.6139, 标准差=0.2482

【贵州荔波】 (共2题)
  相关性:     均值=0.7641, 标准差=0.0231
  可解释性:   均值=1.0000, 标准差=0.0000
  难度准确性: 均值=0.6504, 标准差=0.2083

【广西桂林】 (共5题)
  相关性:     均值=0.6580, 标准差=0.0595
  可解释性:   均值=1.0000, 标准差=0.0000
  难度准确性: 均值=0.6932, 标准差=0.1521

【海积地貌】 (共3题)
  相关性:     均值=0.8275, 标准差=0.0076
  可解释性:   均值=1.0000, 标准差=0.0000
  难度准确性: 均值=0.6457, 标准差=0.1909

【冰川地貌】 (共4题)
  相关性:     均值=0.7959, 标准差=0.0462
  可解释性

In [76]:
# 批量分析文件夹中所有评估结果文件
import os
import glob

def analyze_folder(folder_path, pattern='*_rel_ans_dif.txt'):
    """
    批量分析文件夹中所有评估结果文件，汇总报告
    """
    search_path = os.path.join(folder_path, pattern)
    files = glob.glob(search_path)

    if not files:
        print(f"未找到匹配文件: {search_path}")
        return

    print("="*80)
    print(f"批量分析: {folder_path}")
    print(f"找到 {len(files)} 个文件")
    print("="*80)

    results = []

    for txt_path in sorted(files):
        with open(txt_path, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f if line.strip()]

        all_rel, all_exp, all_acc = [], [], []

        for line in lines:
            parts = line.split(';')
            if len(parts) < 16:
                continue

            # 相关性在索引9，5次作答在10-14，难度准确性在15
            all_rel.append(float(parts[9]) if parts[9] else 0.0)

            answers = parts[10:15]
            valid_answers = [a for a in answers if a and a in 'ABCD']
            if valid_answers:
                from collections import Counter
                counter = Counter(valid_answers)
                most_common_count = counter.most_common(1)[0][1]
                exp_score = most_common_count / len(valid_answers)
            else:
                exp_score = 0.0
            all_exp.append(exp_score)

            all_acc.append(float(parts[15]) if parts[15] else 0.0)

        filename = os.path.basename(txt_path)
        results.append({
            'filename': filename,
            'n': len(all_rel),
            'relevance': np.mean(all_rel) if all_rel else 0,
            'explainability': np.mean(all_exp) if all_exp else 0,
            'accuracy': np.mean(all_acc) if all_acc else 0
        })

    print(f"\n{'文件名':<40} {'题数':>6} {'相关性':>10} {'可解释性':>10} {'难度准确性':>10}")
    print("-"*80)
    for r in results:
        print(f"{r['filename']:<40} {r['n']:>6} {r['relevance']:>10.4f} {r['explainability']:>10.4f} {r['accuracy']:>10.4f}")

    print("-"*80)
    avg_rel = np.mean([r['relevance'] for r in results])
    avg_exp = np.mean([r['explainability'] for r in results])
    avg_acc = np.mean([r['accuracy'] for r in results])
    print(f"{'各指标平均值':<40} {'':>6} {avg_rel:>10.4f} {avg_exp:>10.4f} {avg_acc:>10.4f}")


批量分析: ./output/comparison/analysis/
找到 2 个文件

文件名                                          题数        相关性       可解释性      难度准确性
--------------------------------------------------------------------------------
ds-210-noncontext_rel_ans_dif.txt           100     0.6517     0.9840     0.7751
ds-210-text-rag_rel_ans_dif.txt             100     0.7205     0.9960     0.7925
--------------------------------------------------------------------------------
各指标平均值                                              0.6861     0.9900     0.7838


In [89]:
source = "./output/comparison/raw"   # 替换为你的起始文件夹
target = "./output/comparison/analysis"     # 替换为目标文件夹
copy_files(source, target)

已复制: ./output/comparison/raw\ds-210-noncontext_rel_ans_dif.txt -> ./output/comparison/analysis\ds-210-noncontext_rel_ans_dif.txt
已复制: ./output/comparison/raw\ds-210-text-rag_rel_ans_dif.txt -> ./output/comparison/analysis\ds-210-text-rag_rel_ans_dif.txt
已复制: ./output/comparison/raw\ds-220-noncontext_rel_ans_dif.txt -> ./output/comparison/analysis\ds-220-noncontext_rel_ans_dif.txt
已复制: ./output/comparison/raw\ds-220-text-rag_rel_ans_dif.txt -> ./output/comparison/analysis\ds-220-text-rag_rel_ans_dif.txt
已复制: ./output/comparison/raw\ds-230-noncontext_rel_ans_dif.txt -> ./output/comparison/analysis\ds-230-noncontext_rel_ans_dif.txt
已复制: ./output/comparison/raw\ds-230-text-rag_rel_ans_dif.txt -> ./output/comparison/analysis\ds-230-text-rag_rel_ans_dif.txt
已复制: ./output/comparison/raw\questions_210_qwen_rel_ans_dif.txt -> ./output/comparison/analysis\questions_210_qwen_rel_ans_dif.txt
已复制: ./output/comparison/raw\questions_220_ds_rel_ans_dif.txt -> ./output/comparison/analysis\questions_220

In [90]:
# 批量分析文件夹里的多批样本
analyze_folder('./output/comparison/analysis')

批量分析: ./output/comparison/analysis
找到 16 个文件

文件名                                          题数        相关性       可解释性      难度准确性
--------------------------------------------------------------------------------
ds-210-noncontext_rel_ans_dif.txt           100     0.6517     0.9840     0.7751
ds-210-text-rag_rel_ans_dif.txt             100     0.7205     0.9960     0.7925
ds-220-noncontext_rel_ans_dif.txt           100     0.6324     0.9660     0.7235
ds-220-text-rag_rel_ans_dif.txt             100     0.6938     0.9940     0.7447
ds-230-noncontext_rel_ans_dif.txt            67     0.6598     0.9701     0.6412
ds-230-text-rag_rel_ans_dif.txt             100     0.7028     0.9820     0.6572
questions_210_qwen_rel_ans_dif.txt           50     0.7308     0.9840     0.7962
questions_220_ds_rel_ans_dif.txt             46     0.6921     0.9826     0.7326
questions_220_qwen_rel_ans_dif.txt           50     0.7192     0.9920     0.7240
questions_230_qwen_rel_ans_dif.txt           50     0.7813     

# 文件操作工具

In [91]:
# 文件筛选转移器 
# 把一个文件夹里符合某种命名的文件，copy到另一个文件夹里

import os
import shutil

def copy_files(src_dir, dst_dir, pattern="_rel_ans_dif.txt"):

    # 确保目标文件夹存在
    os.makedirs(dst_dir, exist_ok=True)

    # 递归遍历起始文件夹
    for root, dirs, files in os.walk(src_dir):
        for file in files:
            if pattern in file:
                src_path = os.path.join(root, file)
                dst_path = os.path.join(dst_dir, file)
                try:
                    shutil.copy2(src_path, dst_path)  # copy2 保留文件元数据
                    print(f"已复制: {src_path} -> {dst_path}")
                except Exception as e:
                    print(f"复制失败 {src_path}: {e}")


In [1]:
# ===================== 评估数据读取函数 =====================
import pandas as pd
import numpy as np
from collections import Counter

def load_eval_data(file_path):
    """
    读取评估结果文件，提取三个指标：
    - 知识相关性：倒数第7个字段
    - 5次作答（用于计算可解释性）：倒数第6到倒数第2个字段
    - 难度准确性：倒数第1个字段
    
    文件格式示例：
    ...;0.8221;A;A;A;A;A;0.7628
    
    参数：
        file_path: txt文件路径，;分割，无表头
    返回：
        DataFrame，最后三列为 relevance, explainability, accuracy
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]
    
    results = []
    for line in lines:
        parts = line.split(';')
        
        # 提取三个指标
        relevance = float(parts[-7])          # 知识相关性
        answers = parts[-6:-1]                # 5次作答字母
        accuracy = float(parts[-1])          # 难度准确性
        
        # 计算可解释性：最常见答案出现次数 / 5
        valid_answers = [a for a in answers if a in 'ABCD']
        if valid_answers:
            counter = Counter(valid_answers)
            most_common_count = counter.most_common(1)[0][1]
            explainability = most_common_count / len(valid_answers)
        else:
            explainability = 0.0
        
        # 保留其他字段，加上三个指标
        other_fields = parts[:-7]
        new_row = other_fields + [relevance, explainability, accuracy]
        results.append(new_row)
    
    # 构建列名
    n_other = len(results[0]) - 3
    col_names = [f'field_{i}' for i in range(n_other)] + ['relevance', 'explainability', 'accuracy']
    
    df = pd.DataFrame(results, columns=col_names)
    return df

# -------- 使用示例 --------
# df = load_eval_data(r'C:/Users/lenovo/LLM/thesis/geograph/output/comparison/组合数据/你的文件名.txt')
# print(df[['relevance', 'explainability', 'accuracy']].describe())


In [4]:
# ===================== 评估数据读取函数 =====================
import pandas as pd
import numpy as np
from collections import Counter

def load_eval_data(input_path, output_path=None):
    """
    读取评估结果文件，计算可解释性，替换最后7个字段为3个指标，保存新文件
    
    输入格式示例：
    ...;0.8221;A;A;A;A;A;0.7628
    (倒数第7=知识相关性，倒数第6~2=5次作答，倒数第1=难度准确性)
    
    输出格式：
    ...;0.8221;1.0000;0.7628
    (倒数第3=知识相关性，倒数第2=可解释性，倒数第1=难度准确性)
    
    参数：
        input_path: 输入文件路径，;分割，无表头
        output_path: 输出文件路径，默认为输入文件名加 '_metrics' 后缀
    """
    if output_path is None:
        base = input_path.rsplit('.', 1)
        output_path = base[0] + '_metrics.' + base[1]
    
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]
    
    results = []
    for line in lines:
        parts = line.split(';')
        
        # 提取三个指标
        relevance = float(parts[-7])          # 知识相关性
        answers = parts[-6:-1]              # 5次作答字母
        accuracy = float(parts[-1])          # 难度准确性
        
        # 计算可解释性
        valid_answers = [a for a in answers if a in 'ABCD']
        if valid_answers:
            counter = Counter(valid_answers)
            most_common_count = counter.most_common(1)[0][1]
            explainability = most_common_count / len(valid_answers)
        else:
            explainability = 0.0
        
        # 前部字段不变 + 3个指标
        other_fields = parts[:-7]
        new_row = other_fields + [f'{relevance:.4f}', f'{explainability:.4f}', f'{accuracy:.4f}']
        results.append(';'.join(new_row))
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(''.join(results))
    
    print(f'已保存至：{output_path}')
    return output_path

# -------- 使用示例 --------
# load_eval_data(r'C:/Users/lenovo/LLM/thesis/geograph/output/comparison/组合数据/你的文件名.txt')


In [6]:
# ===================== 评估数据读取函数 =====================
import pandas as pd
import numpy as np
from collections import Counter

def load_eval_data(input_path, output_path=None):
    """
    读取评估结果文件，计算可解释性，替换最后7个字段为3个指标，保存新文件
    
    输入格式示例：
    ...;0.8221;A;A;A;A;A;0.7628
    (倒数第7=知识相关性，倒数第6~2=5次作答，倒数第1=难度准确性)
    
    输出格式：
    ...;0.8221;1.0000;0.7628
    (倒数第3=知识相关性，倒数第2=可解释性，倒数第1=难度准确性)
    
    参数：
        input_path: 输入文件路径，;分割，无表头
        output_path: 输出文件路径，默认为输入文件名加 '_metrics' 后缀
    """
    if output_path is None:
        base = input_path.rsplit('.', 1)
        output_path = base[0] + '_metrics.' + base[1]
    
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]
    
    results = []
    for line in lines:
        parts = line.split(';')
        
        # 提取三个指标
        relevance = float(parts[-7])          # 知识相关性
        answers = parts[-6:-1]              # 5次作答字母
        accuracy = float(parts[-1])          # 难度准确性
        
        # 计算可解释性
        valid_answers = [a for a in answers if a in 'ABCD']
        if valid_answers:
            counter = Counter(valid_answers)
            most_common_count = counter.most_common(1)[0][1]
            explainability = most_common_count / len(valid_answers)
        else:
            explainability = 0.0
        
        # 前部字段不变 + 3个指标
        other_fields = parts[:-7]
        new_row = other_fields + [f'{relevance:.4f}', f'{explainability:.4f}', f'{accuracy:.4f}']
        results.append(';'.join(new_row))
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(results))
    
    print(f'已保存至：{output_path}')
    return output_path

# -------- 使用示例 --------
# load_eval_data(r'C:/Users/lenovo/LLM/thesis/geograph/output/comparison/组合数据/你的文件名.txt')


In [12]:
load_eval_data('C:/Users/lenovo/LLM/thesis/geograph/output/comparison/组合数据/qwen图检索3指标.txt')

已保存至：C:/Users/lenovo/LLM/thesis/geograph/output/comparison/组合数据/qwen图检索3指标_metrics.txt


'C:/Users/lenovo/LLM/thesis/geograph/output/comparison/组合数据/qwen图检索3指标_metrics.txt'